# 06 - Full pipeline on Kaggle: demo results + final evaluation
Runs the complete proposed pipeline on a Kaggle GPU:

**Attention U-Net → damage-aware module → Stage 1 (SD 1.5 + mural LoRA) → Stage 2 (Canny ControlNet + Canny LoRA) → final diffusion → post-processing**

Part A restores 6 **real damaged murals** from the MuralDH `Mural_seg` test set and saves every stage, so the Streamlit app on a CPU laptop can show them instantly (`demo_samples.zip`).
Part B evaluates the pipeline stage by stage with PSNR / SSIM / LPIPS (`pipeline_eval.json`).

**Kaggle setup**
1. Upload the project folder (this repository) as a new Kaggle dataset (name e.g. `mural-final`). It must contain `models/attention_unet.keras`, `models/dunhuang_inpaint_lora.safetensors`, `models/dunhuang_canny_lora.safetensors`.
2. Add Input: `MuralDH` and `mural-final`.
3. Accelerator **GPU T4**, Internet **ON**. Run All (about 30-40 min).


In [ ]:
!pip install -q lpips peft diffusers transformers accelerate safetensors scikit-image opencv-python-headless


In [ ]:
import os, sys, glob, shutil, json, time, random, zipfile
import numpy as np, cv2
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
tf.config.set_visible_devices([], "GPU")          # segmentation on CPU, so PyTorch keeps the whole GPU for diffusion
try:                                                # Kaggle's torchao is too old for peft; we do not use it
    import peft.tuners.lora.torchao as _pt; _pt.is_torchao_available = lambda: False
except Exception as e: print("torchao patch skipped:", e)
import torch
assert torch.cuda.is_available(), "Turn on GPU T4"

# ---- project folder (the uploaded mural_final dataset) -> writable copy
cands = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.py", recursive=True)
         if os.path.isdir(os.path.join(os.path.dirname(p), "model")) and os.path.isdir(os.path.join(os.path.dirname(p), "tools"))]
assert cands, "project dataset not found - upload this repository as a Kaggle dataset and add it as input"
PROJECT = "/kaggle/working/mural_final"
if os.path.exists(PROJECT): shutil.rmtree(PROJECT)
shutil.copytree(cands[0], PROJECT)
for d in ("model", "tools", "utils", "evaluation", "tests"):
    if os.path.isdir(f"{PROJECT}/{d}"): open(f"{PROJECT}/{d}/__init__.py", "a").close()
sys.path.insert(0, PROJECT); os.chdir(PROJECT)
os.environ["MURAL_DEVICE"] = "cuda"
import config
from model.inference import restore_mural
from model.diffusion_model import classical_inpaint
from model.postprocess import composite
from evaluation.restoration_metrics import restoration_metrics
SEG = config.SEGMENTATION_MODELS[config.DEFAULT_SEGMENTATION_MODEL]["path"]
print("Attention U-Net:", SEG.is_file(), "| mural LoRA:", config.LORA_PATH.is_file(), "| Canny LoRA:", config.CANNY_LORA_PATH.is_file())

# ---- MuralDH
SEG_ROOT = Path([os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/Mural_seg/**/test/images", recursive=True)][0])
CLEAN_DIR = Path(os.path.dirname(glob.glob("/kaggle/input/**/img_0crop_0_0.png", recursive=True)[0]))
print("Mural_seg:", SEG_ROOT, "| Mural512:", CLEAN_DIR)

def read_label(lab_dir, fname):
    p = lab_dir / fname
    if not p.exists(): p = next(iter(lab_dir.glob(Path(fname).stem + ".*")))
    return cv2.imread(str(p), cv2.IMREAD_GRAYSCALE) > 0
CFG = {"restore_mode": "full", "steps": 30, "seed": 42, "damage_aware": True}


## Part A - demo results on real damaged murals

In [ ]:
# pick 6 real damaged murals from the TEST split: visible damage (3-30 % of the image), spread over the set
test_files = sorted(os.listdir(SEG_ROOT / "test" / "images"))
cov = {f: float(read_label(SEG_ROOT / "test" / "labels", f).mean() * 100) for f in test_files}
ok = [f for f in test_files if 3 <= cov[f] <= 30]
DEMO = [ok[int(i)] for i in np.linspace(0, len(ok) - 1, 6)]
print("candidates:", len(ok), "| chosen:", [(f, round(cov[f], 1)) for f in DEMO])

SAMPLES = Path(PROJECT) / "data" / "samples"; CACHE = SAMPLES / "cache"
for p in SAMPLES.glob("*.png"): p.unlink()
CACHE.mkdir(parents=True, exist_ok=True)
for f in DEMO:
    img = Image.open(SEG_ROOT / "test" / "images" / f).convert("RGB")
    if max(img.size) > 1024: img.thumbnail((1024, 1024), Image.Resampling.LANCZOS)
    img.save(SAMPLES / f"real_{Path(f).stem}.png")


In [ ]:
from model.damage_detector import predict_damage_mask
use_unet = SEG.is_file()
print("mask source:", "Attention U-Net prediction" if use_unet else "ground-truth label (attention_unet.keras not found)")
demo_rows = []
for p in sorted(SAMPLES.glob("*.png")):
    image = Image.open(p).convert("RGB"); t = time.time()
    if use_unet:
        mask, prob, stats = predict_damage_mask(image)
        thr = stats.get("threshold", 0.5)
    else:
        lab = read_label(SEG_ROOT / "test" / "labels", p.stem.replace("real_", "", 1) + ".png")
        mask = Image.fromarray((lab * 255).astype(np.uint8)).resize(image.size, Image.Resampling.NEAREST); prob = None; thr = 0.5
    res = restore_mural(image, mask, config_dict={**CFG, "threshold": thr}, prob_map=prob)
    mask.save(CACHE / f"{p.stem}__mask.png")
    res.restored_image.save(CACHE / f"{p.stem}__full.png")
    for k, im in res.stages.items():
        if k in ("types", "prefill", "stage1", "stage2", "final_diffusion", "final"):
            im.save(CACHE / f"{p.stem}__full__{k}.png")
    keep = {k: v for k, v in res.metrics.items() if isinstance(v, (int, float, str, bool, list, dict, type(None)))}
    (CACHE / f"{p.stem}__full__metrics.json").write_text(json.dumps(keep, default=str, indent=1))
    errs = {k: v for k, v in res.metrics.items() if k.endswith("_error") and v}
    demo_rows.append((p.stem, image, res))
    print(f"{p.name}: {time.time()-t:.0f}s | damage {np.asarray(mask).mean()/2.55:.1f}% | "
          f"mural LoRA {res.metrics.get('lora_applied')} | Canny LoRA {res.metrics.get('canny_lora_applied')} | "
          f"tiers {res.metrics.get('damage_summary', {}).get('tiers_used')} | {errs or 'no errors'}")


In [ ]:
show = [("types", "damage types"), ("stage1", "Stage 1"), ("stage2", "Stage 2"), ("final_diffusion", "final diffusion"), ("final", "post-processed")]
fig, ax = plt.subplots(len(demo_rows), len(show) + 1, figsize=(3 * (len(show) + 1), 3 * len(demo_rows)))
for r, (stem, img, res) in enumerate(demo_rows):
    ax[r, 0].imshow(img); ax[r, 0].set_title("damaged input" if r == 0 else "", fontsize=9)
    for c, (k, t) in enumerate(show, start=1):
        if k in res.stages: ax[r, c].imshow(res.stages[k])
        ax[r, c].set_title(t if r == 0 else "", fontsize=9)
    for a in ax[r]: a.axis("off")
plt.tight_layout(); os.makedirs("/kaggle/working/results", exist_ok=True)
plt.savefig("/kaggle/working/results/demo_real_murals.png", dpi=130); plt.show()

with zipfile.ZipFile("/kaggle/working/demo_samples.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in SAMPLES.rglob("*"):
        if p.is_file(): z.write(p, p.relative_to(SAMPLES.parent))     # -> samples/... inside the zip
print("saved /kaggle/working/demo_samples.zip")


## Part B - stage-by-stage evaluation
A restoration can only be scored against the original, and real damaged murals have no original. So the **real damage masks** from `Mural_seg/train/labels` are laid over **clean Mural512 murals**: the damage shapes are real, and the clean mural is the answer key.

Variants (all from the same pipeline run where possible):
| id | output |
|---|---|
| A | classical Telea inpainting (baseline) |
| B | after Stage 1 (SD 1.5 + mural LoRA) |
| C | after Stage 2 (Canny ControlNet + Canny LoRA) |
| D | after the final diffusion |
| E | after post-processing = **full proposed pipeline** |
| F | full pipeline **without** the damage-aware module |

Note: the two LoRAs were trained on all 5,096 Mural512 murals, so these murals were seen in training (as clean images, never with these masks). Report this in the thesis.


In [ ]:
N_EVAL = 30
rng = random.Random(0)
clean_files = sorted(p.name for p in CLEAN_DIR.glob("img_*crop_*_*.png"))
eval_clean = rng.sample(clean_files, N_EVAL)
train_lab_dir = SEG_ROOT / "train" / "labels"
real_masks = []
for f in rng.sample(sorted(os.listdir(train_lab_dir)), 400):
    m = cv2.resize((read_label(train_lab_dir, f) * 255).astype(np.uint8), (512, 512), interpolation=cv2.INTER_NEAREST) > 127
    if 3 <= m.mean() * 100 <= 35: real_masks.append((f, m))
    if len(real_masks) == N_EVAL: break
print("eval pairs:", len(eval_clean), "clean murals x", len(real_masks), "real damage masks")

def make_damaged(clean, m, seed):
    r = np.random.default_rng(seed)
    plaster = np.array([214, 202, 180], np.float32) + r.normal(0, 8, clean.shape)
    out = clean.astype(np.float32); out[m] = plaster[m]
    return Image.fromarray(np.clip(out, 0, 255).astype(np.uint8))

rows, grids, t0 = [], [], time.time()
for i, (cf, (mf, m)) in enumerate(zip(eval_clean, real_masks)):
    clean = Image.open(CLEAN_DIR / cf).convert("RGB").resize((512, 512))
    damaged = make_damaged(np.asarray(clean), m, i); mask = Image.fromarray((m * 255).astype(np.uint8))
    base = {**CFG, "refine_mask": False, "max_side": 512}
    A = composite(damaged, classical_inpaint(damaged, mask), mask)
    full = restore_mural(damaged, mask, config_dict=base)
    nod = restore_mural(damaged, mask, config_dict={**base, "damage_aware": False})
    V = {"damaged": damaged, "A_classical": A, "B_stage1": full.stages.get("stage1"), "C_stage2": full.stages.get("stage2"),
         "D_final_diffusion": full.stages.get("final_diffusion"), "E_full_pipeline": full.restored_image,
         "F_full_without_damage_aware": nod.restored_image}
    for name, out in V.items():
        if out is not None:
            rows.append(dict(image=cf, mask=mf, variant=name, **restoration_metrics(clean, out, mask)))
    if i < 5: grids.append((clean, V))
    print(f"{i+1}/{len(real_masks)}  {time.time()-t0:.0f}s  damage {m.mean()*100:.1f}%", flush=True)


In [ ]:
import pandas as pd
from scipy import stats
df = pd.DataFrame(rows); df.to_csv("/kaggle/working/results/pipeline_per_image.csv", index=False)
cols = ["psnr", "ssim", "psnr_masked", "ssim_masked", "lpips", "lpips_masked"]
order = ["damaged", "A_classical", "B_stage1", "C_stage2", "D_final_diffusion", "E_full_pipeline", "F_full_without_damage_aware"]
summary = df.groupby("variant")[cols].mean().round(4).reindex([o for o in order if o in set(df.variant)])
display(summary)
pairs = {}
for a, b in [("B_stage1", "A_classical"), ("C_stage2", "B_stage1"), ("D_final_diffusion", "C_stage2"),
             ("E_full_pipeline", "D_final_diffusion"), ("E_full_pipeline", "F_full_without_damage_aware"), ("E_full_pipeline", "A_classical")]:
    for met in ("lpips_masked", "psnr_masked", "ssim_masked"):
        x = df[df.variant == a].set_index("image")[met]; y = df[df.variant == b].set_index("image")[met].reindex(x.index)
        d = (x - y).dropna()
        t = stats.ttest_rel(x.loc[d.index], y.loc[d.index]) if len(d) > 2 else None
        pairs[f"{a} vs {b} [{met}]"] = {"mean_diff": float(d.mean()), "p_value": float(t.pvalue) if t is not None else None, "n": int(len(d))}
for k, v in pairs.items(): print(f"{k:70s} {v['mean_diff']:+.4f}  p={v['p_value']:.4f}")
out = {"n_images": int(df.image.nunique()), "masks": "real Mural_seg train labels on clean Mural512 murals",
       "eval_murals_seen_in_lora_training": True, "steps": CFG["steps"],
       "variants": {v: {c: {"mean": float(df[df.variant == v][c].mean()), "std": float(df[df.variant == v][c].std())} for c in cols}
                    for v in order if v in set(df.variant)}, "paired_tests": pairs}
json.dump(out, open("/kaggle/working/results/pipeline_eval.json", "w"), indent=2)

show = order[:]
fig, ax = plt.subplots(len(grids), len(show) + 1, figsize=(2.3 * (len(show) + 1), 2.3 * len(grids)))
for r, (clean, V) in enumerate(grids):
    ax[r, 0].imshow(clean); ax[r, 0].set_title("clean (answer key)" if r == 0 else "", fontsize=7)
    for c, n in enumerate(show, start=1):
        if V.get(n) is not None: ax[r, c].imshow(V[n])
        ax[r, c].set_title(n if r == 0 else "", fontsize=7)
    for a in ax[r]: a.axis("off")
plt.tight_layout(); plt.savefig("/kaggle/working/results/pipeline_qualitative.png", dpi=140); plt.show()


## Download
- `/kaggle/working/demo_samples.zip` -> unzip and replace the `data\samples` folder of `mural_final` on the demo laptop.
- `/kaggle/working/results/`: `pipeline_eval.json`, `pipeline_per_image.csv`, `pipeline_qualitative.png`, `demo_real_murals.png` (download these for the report).
